# 1 nn.Module

It's the basis for neural networks in PyTorch. nn.Module it's the mother class, the we gonna use the POO to inherit the functions that the mother classe have

We also have the intern dicts, that are for we organize everthing that we put int he `__init__`, like:
1. `._parameters`
2. `._modules`
3. `._buffers`

In [9]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        pass

model = Model()
print(f'Parameters: {model._parameters}')
print(f'Modules: {model._modules}')
print(f'Buffers: {model._buffers}')


Parameters: {}
Modules: {}
Buffers: {}


## 1.2 Parameteres

A function that add and transform data into parameters `nn.Parameters()`

In [10]:
import torch

class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.parameter1 = nn.Parameter(torch.tensor(4.0))


model = Model()
print(f'Parameters: {model._parameters}')

Parameters: {'parameter1': Parameter containing:
tensor(4., requires_grad=True)}


## 1.3 Modules

Are the layers of the model (`nn.Linear`, `nn.Conv2d` and more)

In [23]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer1 = nn.Linear(1,1 )


model = Model()
print(f'Modules: {model._modules}')

Modules: {'layer1': Linear(in_features=1, out_features=1, bias=True)}


## 1.4 Buffers

Are the the place where functions of the model that don't have treinable parameters are allocated (Normalization, Masks and more) `.register_buffer()`

In [22]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.constant = torch.tensor(1.0)
        self.register_buffer('my constant' , self.constant)

model = Model()
print(f'Buffers: {model._buffers}')

Buffers: {'my constant': tensor(1.)}


### WARNING!!
### PyTorch Internal Structure: `nn.Module`, Parameters and Buffers

To understand how PyTorch manages models, it`s essential to distinguish between the objects you define (modules) and the data they contain (parameters and buffers).
 1. The Hierarchy of `nn.Module`
In PyTorch, everything that makes up your model is organized into internal "drawers" inside an `nn.Module` object. When you create a class, PyTorch manages these drawers automatically:

* **`._modules`**: Contains sub-modules defined as attributes.
* **`._parameters`**: Contains tensors that you manually set to `nn.Parameter`.
* **`._buffers`**: Contains tensors that you manually registered as buffers.

---
 2. Where should each thing stay?

 A. Occupying the `._modules` (The most common path)
The vast majority of components of a neural network (such as `nn.Linear`, `nn.Conv2d`, `nn.BatchNorm1d`) **must** be assigned directly as an attribute of its class in `__init__`.

* **Why?** By doing `self.fc = nn.Linear(...)`, PyTorch detects this object and automatically places it in `._modules`.
* **Result:** PyTorch "opens" this module, finds the weights and biases within it, and manages everything automatically (saving, loading, moving between CPU/GPU, and calculating gradients).

 B. Using `._parameters` (Manual Definition)
You should only add items to `._parameters` if you are creating a custom layer from scratch and need a tensor that **should** be updated by the optimizer (via gradient), but is not a ready-made module.

* **How to:** `self.my_param = nn.Parameter(torch.randn(10, 10))`
* **Result:** PyTorch identifies that this is a trainable parameter and includes it in the optimizer calculation.

 C. Using `._buffers` (Non-Trainable State)
You should use buffers for data that is part of the model`s "state" but should **not** be changed by the optimizer.

* **As:** `self.register_buffer(`name`, tensor)`
* **Result:** PyTorch knows that this tensor needs to be saved in `state_dict` and moved along with the model (e.g. `to(`cuda`)`), but the optimizer will not try to apply gradients over it.

---
3. Visual Summary of the Framework

| Location | What do you keep? | When to use? |
| :- | :- | :- |
| **`._modules`** | Layers (nn.Linear, nn.Conv, etc) | Whenever using ready-made PyTorch layers. |
| **`._parameters`** | Manual trainable tensors | If creating custom weight logic. |
| **`._buffers`** | Fixed state tensors | If you need constants, masks, or moving averages. |

4. Why don`t you see the parameters inside `self._parameters`?
If you set `self.fc = nn.Linear(10, 5)`, the weights (`weight`) and bias (`bias`) of the `fc` layer **will not** be in the `self._parameters` of your main class. They will be inside the `_parameters` drawer of the **`self.fc` object**.

To visualize everything in an organized way, PyTorch offers recursive methods:
- `model.named_parameters()`: Lists all parameters of the root and all submodules.
- `model.named_modules()`: List all the modules in the model
- `model.named_buffers()`: Lists all buffers for the root and all submodules.
"""

with open("structure_pytorch.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)



In [38]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ParameterOwn = nn.Parameter(torch.tensor(2.0))
        self.l1 = nn.Linear(3,3)
        self.bn = nn.BatchNorm1d(num_features=2)

model = Model()

print(f'Parameters:')
for name, param in model.named_parameters():
    print(f"Name: {name} | Size: {param.shape} | Trainable?: {param.requires_grad}")

print(f'\nModules:')
for name, module in model.named_modules():
    print(f"`Path: {name} | Module: {module}")

print(f'\nBuffers:')
for name, buffer in model.named_buffers():
    print(f"Name: {name} | Size: {buffer.shape}")


Parameters:
Name: ParameterOwn | Size: torch.Size([]) | Trainable?: True
Name: l1.weight | Size: torch.Size([3, 3]) | Trainable?: True
Name: l1.bias | Size: torch.Size([3]) | Trainable?: True
Name: bn.weight | Size: torch.Size([2]) | Trainable?: True
Name: bn.bias | Size: torch.Size([2]) | Trainable?: True

Modules:
`Path:  | Module: Model(
  (l1): Linear(in_features=3, out_features=3, bias=True)
  (bn): BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
`Path: l1 | Module: Linear(in_features=3, out_features=3, bias=True)
`Path: bn | Module: BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

Buffers:
Name: bn.running_mean | Size: torch.Size([2])
Name: bn.running_var | Size: torch.Size([2])
Name: bn.num_batches_tracked | Size: torch.Size([])


## 1.5 Acess the Weights

Now that we see how to saw and understand the parameters, modules and buffers from the model. We need to have the acess of that values. Using the: 
- `state_dict().keys()`: Collects all parameters (weights/bias) and all buffers (moving averages, etc.) from the entire model, including submodules, into a single Python dictionary.


In [ ]:
state = model.state_dict()
l1 = state['l1.weight']

print(f'All the weights: {state}')
print(f'Acess the L1 Weights: {l1}')

All the weights: OrderedDict({'ParameterOwn': tensor(2.), 'l1.weight': tensor([[-0.5115, -0.3357, -0.1735],
        [-0.4516, -0.1347,  0.0867],
        [ 0.3745, -0.3747, -0.2883]]), 'l1.bias': tensor([-0.1719,  0.3480, -0.4418]), 'bn.weight': tensor([1., 1.]), 'bn.bias': tensor([0., 0.]), 'bn.running_mean': tensor([0., 0.]), 'bn.running_var': tensor([1., 1.]), 'bn.num_batches_tracked': tensor(0)})
Acess the L1 Weights: tensor([[-0.5115, -0.3357, -0.1735],
        [-0.4516, -0.1347,  0.0867],
        [ 0.3745, -0.3747, -0.2883]])
